# COS30049 A2 - Explorer Assistant 3B, round 2: QLoRA fine-tune of Qwen2.5-3B-Instruct on both T4s

Round 2 of the Qwen writer (`docs/KAGGLE_LM_PLAN.md`): a fresh adapter on about four times round 1's explanation
data, the glossary repeated 4 times and 10,000 DS6 conversations (`configs/qwen3b_explain_v2.json`), at most 2 epochs,
stopping early when the validation loss stops improving. It trains on **both** GPUs of a T4 x2 session (torchrun,
`lm/dist.py`); round 1 used one, which left half of every quota hour idle.

**Settings (right-hand panel):** Accelerator **GPU T4 x2**, **Internet on**. **Inputs:** this account's notebooks
`cos30049a2-qwen-data` (the data) and `cos30049a2-qwen3b-explain` (round 1, whose adapter is also scored on the new
glossary test). Then **Save Version -> Save & Run All (Commit)**; the page can be closed.

**What it does:** clones the repository at `COMMIT`, installs `environment-gpu.yml`'s versions, copies the data from
the input and checks every file's SHA-256 against its `data_manifest.json`, benchmarks 5 steps, trains until done or
until the session's time is nearly up, and then, if at least `EVAL_HOURS` are left, evaluates with both GPUs at once:
round 2 on the frozen DS7 v1 test, DS10 and the glossary test, round 1 on the glossary test.

**If the session ends first:** add this notebook's own previous output as an input, set `RESUME_FROM` to its
`runs/qwen3b-explain-v2` folder, and run again: training continues from the last checkpoint (at most 20 minutes are
lost), or, if it had finished, only the evaluation runs.

**Bringing the result home:** `kaggle kernels output <owner>/cos30049a2-qwen3b-explain-v2 -p <dir>`, then copy
`runs/qwen3b-explain-v2/` to `runs/` and `reports/tables/assistant_eval*` to `reports/tables/`.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "11973ad51a741ee4051b28b90a4e7d5a6295a767"   # the code this run uses; recorded in kaggle_run.json
PINS = ["transformers==4.52.4", "peft==0.15.2", "bitsandbytes==0.48.1", "accelerate==1.7.0"]  # environment-gpu.yml
SESSION_HOURS = 11.75  # Kaggle ends a GPU session at 12 h; everything here finishes before this
EVAL_HOURS = 1.0       # what the evaluations need with both GPUs and EVAL_BATCH (training stops early enough)
EVAL_BATCH = 16        # prompts Qwen answers together
RESUME_FROM = None     # e.g. "/kaggle/input/<this notebook>/runs/qwen3b-explain-v2" to continue a stopped run

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU in this session: Settings panel -> Accelerator -> GPU T4 x2 (needs a verified phone).")
GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
RUN = WORK / "runs" / "qwen3b-explain-v2"

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

sh("nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv")
print(f"{GPUS} GPU(s)")

In [ ]:
def find_input(rel):
    """The file under /kaggle/input whose path ends with rel (inputs mount under their own folder names)."""
    hits = [p for p in Path("/kaggle/input").rglob(Path(rel).name) if p.as_posix().endswith(rel)]
    if not hits:
        raise SystemExit(f"{rel} is not in any input: add the notebooks named at the top as inputs")
    return hits[0]


# The data notebook's output, file by file, checked against its manifest.
manifest_path = find_input("data_manifest.json")
data_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
for rel, digest in data_manifest["files"].items():
    dest = SRC / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(manifest_path.parent / rel, dest)
    assert sha256(dest) == digest, f"{rel} differs from data_manifest.json; stop and check"
print(f"{len(data_manifest['files'])} data files match data_manifest.json (built at commit {data_manifest['commit'][:7]})")

cfg = json.loads((SRC / "configs" / "qwen3b_explain_v2.json").read_text(encoding="utf-8"))
cfg["out_dir"] = str(RUN)
CONFIG = Path("/tmp/qwen3b_v2_kaggle.json")
CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
V1_ADAPTER = find_input("runs/qwen3b-explain/adapter_best/adapter_config.json").parent
if RESUME_FROM and not (RUN / "state.pt").exists():
    shutil.copytree(RESUME_FROM, RUN, dirs_exist_ok=True)
    print("resuming from", RESUME_FROM)
TORCHRUN = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", str(GPUS)]

In [ ]:
# 5 optimizer steps on the longest batches, on every GPU: speed and peak memory, nothing saved (skipped when resuming).
if not (RUN / "state.pt").exists():
    sh(*TORCHRUN, "-m", "lm.qlora", "--config", str(CONFIG), "--bench", log="qwen-v2-bench.log")

In [ ]:
def finished():
    meta_path = RUN / "meta.json"
    if not meta_path.exists():
        return False
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    return meta["steps_done"] >= meta["max_steps"] or bool(meta.get("stopped"))


# Train for the time the session has left, keeping EVAL_HOURS (and a margin for loading) for the evaluation.
hours = SESSION_HOURS - (time.time() - T0) / 3600 - EVAL_HOURS - 0.3
if not finished():
    sh(*TORCHRUN, "-m", "lm.qlora", "--config", str(CONFIG), "--max-hours", f"{hours:.2f}", log="qwen-v2-train.log")
print("training finished" if finished() else "training stopped for time: run again with RESUME_FROM (see the top)")

In [ ]:
# Every evaluation the adoption rule needs (docs/KAGGLE_LM_PLAN.md section 3), shared between the GPUs: one queue
# of commands per GPU, run side by side. Different test sets write different tables, so they never collide.
V1_CONFIG = SRC / "configs" / "qwen3b_explain.json"
GLOSSARY = SRC / "data" / "processed" / "ds7_v2" / "glossary_test.jsonl"
EVALS = [("qwen3b_v2", CONFIG, RUN / "adapter_best", SRC / "data" / "processed" / "ds7" / "test.jsonl", ""),
         ("qwen3b_v2", CONFIG, RUN / "adapter_best", SRC / "data" / "processed" / "ds10" / "explain.jsonl", "ds10"),
         ("qwen3b_v2", CONFIG, RUN / "adapter_best", GLOSSARY, "glossary"),
         ("qwen3b", V1_CONFIG, V1_ADAPTER, GLOSSARY, "glossary")]
# The last two write one table (assistant_eval_glossary.csv), so they share a queue.
QUEUES = [[EVALS[0]], EVALS[1:]] if GPUS > 1 else [EVALS]


def run_queue(gpu, queue):
    for model, config, adapter, test, tag in queue:
        sh(sys.executable, "-m", "lm.eval_explain", "--model", model, "--config", str(config), "--adapter", str(adapter),
           "--test", str(test), "--tag", tag, "--batch-size", str(EVAL_BATCH),
           log=f"eval-{model}-{tag or 'ds7'}.log", env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))


left = SESSION_HOURS - (time.time() - T0) / 3600
if not finished():
    print("not evaluated: training has not finished")
elif left < EVAL_HOURS:
    print(f"not evaluated: {left:.1f} h left; run again with RESUME_FROM and it only evaluates")
else:
    threads = [threading.Thread(target=run_queue, args=(gpu, q)) for gpu, q in enumerate(QUEUES)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    tables = WORK / "reports" / "tables"
    tables.mkdir(parents=True, exist_ok=True)
    for f in (SRC / "reports" / "tables").glob("assistant_eval*.csv"):
        shutil.copy(f, tables)

In [ ]:
# What ran, where, with which versions: for the report's reproducibility notes.
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
meta = json.loads((RUN / "meta.json").read_text(encoding="utf-8")) if (RUN / "meta.json").exists() else {}
(WORK / "kaggle_run.json").write_text(json.dumps({
    "commit": COMMIT, "data_commit": data_manifest["commit"], "gpus": gpu, "python": sys.version.split()[0],
    "packages": {p: version(p) for p in ("torch", "transformers", "peft", "bitsandbytes", "accelerate")},
    "finished": finished(), "resumed_from": RESUME_FROM, "session_hours": round((time.time() - T0) / 3600, 2),
    "meta": {k: meta.get(k) for k in ("steps_done", "max_steps", "best_val_loss", "stopped", "tok_per_s",
                                      "peak_memory_gb", "hardware", "fp16_loss_scaling")},
}, indent=2), encoding="utf-8")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 2**20:9.1f} MB  {p.relative_to(WORK)}")